In [1]:
import os
import cv2 as cv
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

Kode di atas di gunakan untuk mengimport beberapa library yang dibutuhkan untuk mengerjakan project pengolahan citra digital. Adapun beberapa librarynya antara lain: 

* IMPORT OS
  Digunakan untuk berinteraksi dengan sistem operasi, seperti membuat folder, membaca direktori, mengecek keberadaan file, dan mengelola path file.

* IMPORT CV2 AS CV
  Library ini digunakan untuk pengolahan citra digital dan computer vision, seperti membaca gambar, mengubah ukuran citra, konversi warna, deteksi objek, filtering, dan transformasi citra. Pada project kali ini, cv2 dikhususkan untuk pengolahan citra seperti resize, membaca gambar, grayscale dan menyimmpan hasil.

* IMPORT NUMPY AS NP
  NumPy digunakan untuk operasi numerik dan manipulasi array/matriks, yang menjadi dasar representasi citra digital dalam bentuk matriks piksel. Dalam project ini, numpy lebih sering digunakan dalam pembuatan kernel, kanvas kosong, padding, menghitung rata-rata, median filter, mengubah data, membatasi nilai piksel, dan lain sebagainya.

* FROM PATHLIB IMPORT PATH
  Mengimpor kelas Path dari modul pathlib untuk mempermudah pengelolaan path file dan folder secara lebih fleksibel dan kompatibel di berbagai sistem operasi. Dalam project kali ini, path digunakan untuk mendeteksi lokasi project, membuat path, membuat folder output, menyusun path file, dan lain sebagainya.

* IMPORT MATHPLOTLIB.PYPLOT AS PLT
  Digunakan untuk menampilkan gambar, membuat grafik, visualisasi hasil pengolahan citra, serta membandingkan beberapa citra dalam satu tampilan.



In [7]:
kernelSharpening = np.array([
    [1/9, 1/9, 1/9],
    [1/9, 8/9, 1/9],
    [1/9, 1/9, 1/9]
], dtype=np.float32)

sobelX = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float32)

sobelY = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.float32)

kernel_diamond = np.array([
    [0,0,1,0,0],
    [0,1,1,1,0],
    [1,1,1,1,1],
    [0,1,1,1,0],
    [0,0,1,0,0]
])

* Kernel Sobel X

Kernel Sobel X digunakan untuk mendeteksi tepi citra pada arah horizontal atau perubahan intensitas yang terjadi dari kiri ke kanan. Operator ini menghitung gradien pada sumbu X sehingga objek yang memiliki batas vertikal akan tampak lebih jelas.

* Kernel Sobel Y

Kernel Sobel Y digunakan untuk mendeteksi tepi citra pada arah vertikal atau perubahan intensitas dari atas ke bawah. Operator ini menghitung gradien pada sumbu Y sehingga objek dengan batas horizontal dapat teridentifikasi dengan lebih baik.

* Kernel Diamond

Kernel Diamond merupakan struktur elemen pada operasi morfologi citra seperti dilasi dan erosi. Bentuk berlian dipilih untuk memperluas atau mengurangi area objek dengan mempertahankan bentuk yang lebih alami dibandingkan kernel persegi. Nilai 1 menunjukkan area yang diproses sedangkan 0 diabaikan.

In [8]:
def thresholding(img, batas):
    baris, kolom = img.shape
    canvas = np.zeros_like(img, dtype=np.uint8)

    for i in range(baris):
        for j in range(kolom):
            if(img[i,j] > batas):
                canvas[i,j] = 255
            elif(img[i,j] <= batas):
                canvas[i,j] = 0

    return canvas

def filter(img, size, mode):
    # dimensi gambar
    height, width = img.shape

    # ukuran padding
    pad = size // 2

    # tambah padding di sisi gambar
    padded = np.pad(img, pad, mode='edge')

    # canvas hasil
    canvas = np.zeros_like(img, dtype=np.uint8)

    match mode:

        # FILTER RATA-RATA / MEAN FILTER
        case "mean":
            area = size * size

            for i in range(height):
                for j in range(width):

                    # area kernel
                    region = padded[i:i+size, j:j+size]

                    # hitung mean
                    canvas[i, j] = np.sum(region) / area

        # FILTER MEDIAN
        case "median":
            for i in range(height):
                for j in range(width):

                    # area kernel
                    region = padded[i:i+size, j:j+size]

                    # hitung median
                    canvas[i, j] = np.median(region)

        # FILTER MODUS / MODE FILTER
        case "modus":
            for i in range(height):
                for j in range(width):

                    # area kernel
                    region = padded[i:i+size, j:j+size]

                    # flatten array
                    values = region.ravel()

                    # hitung frekuensi
                    count = {}

                    for val in values:
                        if val in count:
                            count[val] += 1
                        else:
                            count[val] = 1

                    # cari nilai terbanyak
                    max_count = 0
                    mode_val = 0

                    for val, freq in count.items():
                        if freq > max_count:
                            max_count = freq
                            mode_val = val

                    # simpan hasil
                    canvas[i, j] = mode_val

    # kembalikan gambar
    return canvas

def convolution(img, kernel):

    # ukuran kernel
    size = kernel.shape[0]

    # ukuran padding
    pad_size = size // 2

    # tambah padding nol
    padded = np.pad(img, pad_size, mode='constant')

    # canvas hasil
    canvas = np.zeros_like(img).astype(np.float32)

    # dimensi gambar
    height, width = img.shape

    # loop baris
    for i in range(height):

        # loop kolom
        for j in range(width):

            # area kernel
            region = padded[i:i+size, j:j+size]

            # hitung konvolusi
            canvas[i, j] = np.sum(region * kernel)

    # kembalikan gambar
    return canvas

def edge_detection(img, kernelX, kernelY):

    # konvolusi arah x
    gx = convolution(img, kernelX)

    # konvolusi arah y
    gy = convolution(img, kernelY)

    # gabungkan gradient
    edge = np.abs(gx) + np.abs(gy)

    # normalisasi ke 0-255
    edge = (edge / edge.max()) * 255

    # ubah ke uint8
    edge = edge.astype(np.uint8)

    return edge

def dilasi(image, kernel):
    height, width = image.shape
    k_height, k_width = kernel.shape
    center = k_height//2
    hasil = np.zeros((height, width))

    for i in range(center, height-center):
        for j in range(center, width-center):
            if image[i,j] == 255:
                for k in range(k_height):
                    for l in range(k_width):
                        if kernel[k,l] == 1:
                            hasil[i+k-center,j+l-center] =255
            else:
                if hasil[i,j] !=255:
                    hasil[i,j] = 0 

    return hasil

def thickening(img, kernel, iterasi=1):
    hasil = img.copy()

    for _ in range(iterasi):
        hasil = dilasi(hasil, kernel)

    return hasil

def ekualisasi(citra):

    height, width = citra.shape

    # Histogram
    hist = np.zeros(256, dtype=int)

    # Hitung histogram citra
    for i in range(height):
        for j in range(width):
            hist[int(citra[i, j])] += 1

    # CDF
    cdf = np.zeros(256, dtype=int)
    cdf[0] = hist[0]

    # Hitung CDF
    for i in range(1, 256):
        cdf[i] = cdf[i - 1] + hist[i]

    # Normalisasi CDF
    cdf_normal = np.round(cdf * 255 / (height * width)).astype(np.uint8)

    # Hasil ekualisasi
    hasil = np.zeros_like(citra, dtype=np.uint8)

    # Terapkan hasil CDF normalisasi
    for i in range(height):
        for j in range(width):
            hasil[i, j] = cdf_normal[int (citra[i, j])]

    return hasil


Fungsi thresholding() digunakan untuk mengubah citra grayscale menjadi citra biner berdasarkan nilai ambang yang diberikan melalui parameter batas. Pada awal fungsi, ukuran citra diambil menggunakan img.shape dan disimpan ke dalam variabel baris dan kolom untuk mengetahui jumlah piksel yang akan diproses. Selanjutnya dibuat variabel canvas menggunakan np.zeros_like() dengan tipe data uint8 sebagai tempat penyimpanan hasil akhir yang memiliki ukuran sama dengan citra asli. Proses thresholding dilakukan menggunakan dua perulangan bersarang untuk membaca setiap piksel. Pada setiap posisi [i,j], nilai intensitas piksel dibandingkan dengan nilai batas. Jika nilai piksel lebih besar dari batas maka piksel pada canvas diisi dengan nilai 255 yang merepresentasikan warna putih, sedangkan jika nilai piksel lebih kecil atau sama dengan batas maka diisi dengan nilai 0 yang merepresentasikan warna hitam. Setelah seluruh piksel diproses, fungsi mengembalikan citra biner hasil thresholding.

Fungsi filter() digunakan untuk melakukan proses penyaringan citra menggunakan tiga metode, yaitu mean filter, median filter, dan modus filter. Fungsi dimulai dengan membaca dimensi gambar melalui img.shape kemudian menghitung ukuran padding menggunakan size // 2. Padding ditambahkan menggunakan np.pad() dengan mode edge, yang berarti nilai piksel pada tepi citra akan diperluas agar ukuran hasil tetap sama setelah proses filtering. Setelah itu dibuat canvas kosong untuk menyimpan hasil penyaringan. Pada mode mean, program mengambil area sesuai ukuran kernel (region) lalu menghitung rata-rata seluruh nilai piksel menggunakan np.sum(region)/area, kemudian menyimpan hasilnya pada posisi piksel yang sedang diproses. Pada mode median, setiap area kernel diproses menggunakan np.median() untuk memperoleh nilai tengah yang kemudian digunakan sebagai nilai piksel baru. Pada mode modus, area kernel diubah menjadi array satu dimensi menggunakan ravel(), kemudian setiap nilai dihitung frekuensi kemunculannya menggunakan dictionary count. Nilai yang memiliki frekuensi tertinggi dipilih sebagai nilai pengganti piksel pada canvas. Setelah seluruh proses selesai, citra hasil filter dikembalikan.

Fungsi convolution() digunakan untuk melakukan operasi konvolusi antara citra dan kernel. Langkah pertama adalah membaca ukuran kernel menggunakan kernel.shape[0], kemudian menghitung ukuran padding menggunakan size // 2. Padding ditambahkan ke citra menggunakan mode constant, sehingga area di luar gambar dianggap bernilai nol. Selanjutnya dibuat canvas bertipe float32 untuk menampung hasil konvolusi. Program kemudian melakukan perulangan pada seluruh piksel citra. Pada setiap posisi, area citra dengan ukuran yang sama seperti kernel diambil menggunakan slicing dan disimpan ke variabel region. Area tersebut kemudian dikalikan elemen demi elemen dengan kernel dan seluruh hasil perkalian dijumlahkan menggunakan np.sum(region * kernel). Nilai hasil penjumlahan disimpan ke posisi yang sesuai pada canvas. Setelah seluruh piksel diproses, hasil konvolusi dikembalikan.

Fungsi edge_detection() digunakan untuk mendeteksi tepi objek menggunakan operator Sobel. Proses dimulai dengan memanggil fungsi convolution() menggunakan kernelX untuk memperoleh gradien arah horizontal dan disimpan pada variabel gx. Selanjutnya dilakukan konvolusi menggunakan kernelY untuk memperoleh gradien arah vertikal yang disimpan pada variabel gy. Kedua hasil gradien digabung menggunakan penjumlahan nilai absolut np.abs(gx) + np.abs(gy) sehingga menghasilkan kekuatan tepi dari kedua arah. Agar hasil berada pada rentang intensitas citra standar, dilakukan normalisasi menggunakan (edge / edge.max()) * 255. Setelah itu hasil dikonversi menjadi tipe data uint8 dan dikembalikan sebagai citra hasil deteksi tepi.

Fungsi dilasi() digunakan untuk melakukan operasi morfologi dilasi yang bertujuan memperbesar area objek pada citra biner. Program terlebih dahulu membaca ukuran citra dan ukuran kernel, kemudian menentukan titik pusat kernel menggunakan k_height // 2. Sebuah array kosong hasil dibuat untuk menyimpan hasil dilasi. Program melakukan perulangan pada area citra dengan memperhatikan batas kernel agar tidak keluar dari ukuran gambar. Jika piksel pada posisi [i,j] bernilai 255, maka program akan memeriksa seluruh elemen kernel. Untuk setiap elemen kernel yang bernilai 1, posisi yang bersesuaian pada hasil diubah menjadi 255 sehingga objek menjadi lebih tebal. Jika piksel awal bukan bagian objek dan posisi hasil belum berubah menjadi putih, maka nilainya tetap 0. Setelah seluruh area diproses, citra hasil dilasi dikembalikan.

Fungsi thickening() digunakan untuk melakukan penebalan objek dengan menerapkan operasi dilasi secara berulang. Fungsi dimulai dengan membuat salinan citra asli menggunakan copy() agar data awal tidak berubah. Kemudian dilakukan perulangan sebanyak nilai iterasi, dan pada setiap iterasi fungsi dilasi() dipanggil menggunakan kernel yang diberikan. Hasil dari setiap iterasi digunakan sebagai masukan untuk iterasi berikutnya sehingga objek menjadi semakin tebal. Setelah jumlah iterasi selesai dilakukan, citra hasil penebalan dikembalikan.

Fungsi ekualisasi() digunakan untuk meningkatkan kontras citra menggunakan metode Histogram Equalization. Proses dimulai dengan membaca ukuran citra dan membuat array hist berukuran 256 untuk menyimpan histogram intensitas. Program kemudian melakukan perulangan untuk menghitung jumlah kemunculan setiap nilai intensitas piksel. Setelah histogram diperoleh, dibuat array cdf untuk menghitung Cumulative Distribution Function (CDF). Nilai CDF dihitung secara bertahap dengan menjumlahkan nilai histogram saat ini dengan nilai CDF sebelumnya. Setelah itu dilakukan normalisasi menggunakan rumus cdf * 255 / (height * width) agar hasil berada pada rentang intensitas 0–255. Selanjutnya dibuat array hasil sebagai tempat penyimpanan citra baru. Setiap piksel pada citra asli dipetakan ulang menggunakan nilai CDF yang telah dinormalisasi sehingga distribusi intensitas menjadi lebih merata. Hasil akhirnya adalah citra dengan kontras yang lebih baik dan detail visual yang lebih jelas.


In [9]:
# Deteksi root project
current_path = Path.cwd()

if (current_path / "dataset").exists():
    PROJECT_ROOT = current_path
elif (current_path.parent / "dataset").exists():
    PROJECT_ROOT = current_path.parent
else:
    raise FileNotFoundError("Folder dataset tidak ditemukan.")

DATASET_DIR = PROJECT_ROOT / "dataset"

data = []
labels = []
file_name = []

valid_extensions = [".jpg", ".jpeg", ".png", ".bmp"]

for sub_folder in os.listdir(DATASET_DIR):
    sub_folder_path = DATASET_DIR / sub_folder

    if not sub_folder_path.is_dir():
        continue

    for filename in os.listdir(sub_folder_path):
        img_path = sub_folder_path / filename

        if img_path.suffix.lower() not in valid_extensions:
            continue

        img = cv.imread(str(img_path))

        if img is None:
            print(f"Gagal membaca gambar: {img_path}")
            continue

        img = img.astype(np.uint8)

        data.append(img)
        labels.append(sub_folder)
        file_name.append(filename)

# Pakai dtype object karena ukuran gambar asli bisa beda-beda
data = np.array(data, dtype=object)
labels = np.array(labels)
file_name = np.array(file_name)

print("Jumlah data:", len(data))
print("Label:", np.unique(labels))

Jumlah data: 140
Label: ['catterpillar' 'snail']


Program ini digunakan untuk membaca dan mengumpulkan dataset citra secara otomatis dari folder dataset. Pada awal proses, program mendeteksi lokasi folder dataset dengan memeriksa apakah folder tersebut berada pada direktori saat ini atau satu tingkat di atasnya. Jika folder tidak ditemukan, program akan menghentikan proses dan menampilkan pesan kesalahan.

Setelah lokasi dataset ditemukan, program membuat tiga list yaitu data, labels, dan file_name untuk menyimpan gambar, label kelas, dan nama file. Program kemudian membaca setiap subfolder di dalam dataset, di mana setiap subfolder dianggap sebagai kategori atau label data. Selanjutnya, setiap file pada subfolder diperiksa ekstensinya dan hanya file gambar dengan format .jpg, .jpeg, .png, dan .bmp yang diproses.

Gambar yang valid dibaca menggunakan cv.imread(), kemudian dikonversi menjadi tipe data uint8 sebelum disimpan ke dalam data. Nama folder disimpan sebagai label dan nama file disimpan pada file_name. Setelah seluruh data selesai dibaca, ketiga list dikonversi menjadi array NumPy. Variabel data menggunakan dtype=object karena ukuran gambar dapat berbeda-beda. Terakhir, program menampilkan jumlah data yang berhasil dimuat beserta daftar label yang tersedia pada dataset.


In [10]:
TARGET_SIZE = (384, 256)

def resize_grayscale(image, target_size=TARGET_SIZE):
    resized = cv.resize(image, target_size)

    if len(resized.shape) == 3:
        gray = cv.cvtColor(resized, cv.COLOR_BGR2GRAY)
    else:
        gray = resized

    return gray.astype(np.uint8)


def prepo1(image):
    gray = resize_grayscale(image)
    return gray


def prepo2(image):
    gray = resize_grayscale(image)
    median = filter(gray, 3, "median")
    return median


def prepo3(image):
    gray = resize_grayscale(image)
    median = filter(gray, 3, "median")
    equ = ekualisasi(median)
    return equ


def prepo4(image):
    gray = resize_grayscale(image)
    median = filter(gray, 3, "median")
    sobel = edge_detection(median, sobelX, sobelY)
    return sobel

def prepo5(image):
    gray = resize_grayscale(image)
    median = filter(gray, 3, "median")
    sobel = edge_detection(median, sobelX, sobelY)
    threshold = thresholding(sobel, 33)
    return threshold

PREPROCESSING_METHODS = {
    "prepo1_resize+grayscale": prepo1,
    "prepo2_resize+grayscale+median": prepo2,
    "prepo3_resize+grayscale+median+equ": prepo3,
    "prepo4_resize+grayscale+median+sobel": prepo4,
    "prepo5_resize+grayscale+median+sobel+thresholding": prepo5
}

Program ini digunakan untuk melakukan preprocessing citra sebelum data digunakan pada tahap analisis atau pelatihan model. Seluruh gambar terlebih dahulu diseragamkan ukurannya menjadi 384 × 256 piksel melalui variabel TARGET_SIZE agar seluruh data memiliki dimensi yang sama.

Fungsi resize_grayscale() digunakan untuk mengubah ukuran gambar menggunakan cv.resize() kemudian mengubah gambar menjadi grayscale apabila gambar masih memiliki tiga channel warna (BGR). Jika gambar sudah grayscale, maka gambar langsung digunakan dan dikembalikan dalam tipe data uint8.

Program menyediakan lima metode preprocessing dengan tahapan yang berbeda. prepo1() hanya melakukan resize dan konversi grayscale. prepo2() menambahkan median filter ukuran 3×3 untuk mengurangi noise pada citra. prepo3() melanjutkan hasil median filter dengan ekualisasi histogram untuk meningkatkan kontras gambar. prepo4() menambahkan proses deteksi tepi menggunakan Sobel setelah median filtering sehingga pola tepi objek menjadi lebih terlihat. Sedangkan prepo5() merupakan tahapan paling lengkap dengan menambahkan thresholding bernilai 33 setelah deteksi tepi untuk menghasilkan citra biner.

Seluruh metode preprocessing kemudian disimpan ke dalam dictionary PREPROCESSING_METHODS sehingga setiap metode dapat dipanggil dengan nama tertentu tanpa perlu menjalankan fungsi secara manual satu per satu.


In [11]:
OUTPUT_DIR = PROJECT_ROOT / "preprocessing_output"

for prepo_name, prepo_function in PREPROCESSING_METHODS.items():
    print(f"\nMemproses: {prepo_name}")

    for i in range(len(data)):
        image = data[i]
        label = labels[i]
        filename = file_name[i]

        processed_image = prepo_function(image)
        processed_image = np.clip(processed_image, 0, 255).astype(np.uint8)

        save_dir = OUTPUT_DIR / prepo_name / label
        save_dir.mkdir(parents=True, exist_ok=True)

        save_path = save_dir / filename

        success = cv.imwrite(str(save_path), processed_image)

        if not success:
            print("Gagal simpan:", save_path)

    print(f"Selesai: {prepo_name}")

print("\nSemua preprocessing selesai.")
print("Hasil disimpan di:", OUTPUT_DIR.resolve())


Memproses: prepo1_resize+grayscale
Selesai: prepo1_resize+grayscale

Memproses: prepo2_resize+grayscale+median
Selesai: prepo2_resize+grayscale+median

Memproses: prepo3_resize+grayscale+median+equ
Selesai: prepo3_resize+grayscale+median+equ

Memproses: prepo4_resize+grayscale+median+sobel
Selesai: prepo4_resize+grayscale+median+sobel

Memproses: prepo5_resize+grayscale+median+sobel+thresholding
Selesai: prepo5_resize+grayscale+median+sobel+thresholding

Semua preprocessing selesai.
Hasil disimpan di: D:\Project-PCD-Kelompok-16\preprocessing_output


Program ini digunakan untuk menjalankan seluruh metode preprocessing citra yang telah dibuat sebelumnya dan menyimpan hasilnya ke dalam folder preprocessing_output. Proses dimulai dengan menentukan lokasi penyimpanan menggunakan variabel OUTPUT_DIR.

Selanjutnya program melakukan perulangan pada setiap metode yang terdapat di dalam PREPROCESSING_METHODS. Untuk setiap metode preprocessing, seluruh gambar pada dataset diproses satu per satu dengan mengambil data gambar, label, dan nama file. Gambar kemudian diproses menggunakan fungsi preprocessing yang sesuai dan hasilnya dibatasi ke rentang nilai 0–255 menggunakan np.clip() lalu dikonversi ke tipe uint8 agar sesuai dengan format citra.

Setelah preprocessing selesai, program membuat folder penyimpanan secara otomatis berdasarkan nama metode preprocessing dan label kelas menggunakan mkdir(parents=True, exist_ok=True). Hasil citra kemudian disimpan menggunakan cv.imwrite() dengan nama file yang sama seperti data asli. Jika proses penyimpanan gagal, program akan menampilkan pesan kesalahan.

Setelah seluruh gambar selesai diproses untuk setiap metode, program menampilkan status bahwa preprocessing telah selesai dan menunjukkan lokasi folder tempat seluruh hasil preprocessing disimpan.


In [12]:
valid_extensions = [".jpg", ".jpeg", ".png", ".bmp"]

for prepo_dir in OUTPUT_DIR.iterdir():
    if not prepo_dir.is_dir():
        continue

    print(f"\n{prepo_dir.name}")

    for class_dir in prepo_dir.iterdir():
        if not class_dir.is_dir():
            continue

        image_files = [
            file for file in class_dir.iterdir()
            if file.suffix.lower() in valid_extensions
        ]

        print(f"- {class_dir.name}: {len(image_files)} gambar")


prepo1_resize+grayscale
- catterpillar: 70 gambar
- snail: 70 gambar

prepo2_resize+grayscale+median
- catterpillar: 70 gambar
- snail: 70 gambar

prepo3_resize+grayscale+median+equ
- catterpillar: 70 gambar
- snail: 70 gambar

prepo4_resize+grayscale+median+sobel
- catterpillar: 70 gambar
- snail: 70 gambar

prepo5_resize+grayscale+median+sobel+thresholding
- catterpillar: 70 gambar
- snail: 70 gambar


Program ini digunakan untuk melakukan validasi dan pengecekan hasil preprocessing yang telah disimpan pada folder preprocessing_output. Pada awal proses, ditentukan daftar format gambar yang diperbolehkan melalui variabel valid_extensions, yaitu .jpg, .jpeg, .png, dan .bmp, sehingga hanya file gambar yang akan dihitung.

Program kemudian melakukan perulangan pada setiap folder preprocessing yang terdapat di dalam OUTPUT_DIR. Setiap folder merepresentasikan satu metode preprocessing yang sebelumnya telah dijalankan. Program memeriksa apakah item yang dibaca merupakan folder menggunakan is_dir(), kemudian menampilkan nama metode preprocessing tersebut.

Selanjutnya dilakukan perulangan kembali untuk membaca setiap folder kelas di dalam folder preprocessing. Program mengambil seluruh file yang memiliki ekstensi sesuai dengan daftar valid_extensions, lalu menghitung jumlah gambar menggunakan len(image_files). Hasil akhirnya ditampilkan dalam bentuk nama kelas dan jumlah gambar yang berhasil tersimpan pada masing-masing kelas.

Proses ini digunakan untuk memastikan bahwa seluruh hasil preprocessing telah tersimpan dengan benar dan jumlah gambar pada setiap kategori sesuai dengan data asli.
